In [1]:
import rasterio

in_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist.tif"

with rasterio.open(in_fp) as src:
    print("width:", src.width)
    print("height:", src.height)
    print("bands:", src.count)
    print("dtype:", src.dtypes)
    print("crs:", src.crs)
    print("res:", src.res)

import sys
import numpy
from osgeo import gdal

print(sys.executable)
print(numpy.__version__)
print(gdal.VersionInfo())

width: 130000
height: 140000
bands: 1
dtype: ('float32',)
crs: EPSG:3035
res: (30.0, 30.0)
/mnt/dss_project/lmandl/anaconda3/envs/backcasting/bin/python
1.26.4
3110400


In [2]:
import sys
import numpy
from osgeo import gdal

print(sys.executable)
print(numpy.__version__)
print(gdal.VersionInfo())

/mnt/dss_project/lmandl/anaconda3/envs/backcasting/bin/python
1.26.4
3110400


In [3]:
import math
import subprocess
import sys

in_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist.tif"
tmp_binary_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/tmp_binary_gt07.tif"
out_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_8conn_5ha_1105.tif"

threshold = 0.8
pixel_size = 30
mmu_ha = 5
connectivity = 8

min_pixels = math.ceil((mmu_ha * 10000) / (pixel_size * pixel_size))
print("Minimum pixels for MMU:", min_pixels)

calc_cmd = [
    sys.executable, "-m", "osgeo_utils.gdal_calc",
    "-A", in_fp,
    "--outfile", tmp_binary_fp,
    f"--calc=(A>{threshold})",
    "--type=Byte",
    "--NoDataValue=0",
    "--co=COMPRESS=LZW",
    "--co=TILED=YES",
    "--co=BIGTIFF=YES",
    "--overwrite"
]
subprocess.run(calc_cmd, check=True)

sieve_cmd = [
    sys.executable, "-m", "osgeo_utils.gdal_sieve",
    "-st", str(min_pixels),
    "-8" if connectivity == 8 else "-4",
    tmp_binary_fp,
    out_fp
]
subprocess.run(sieve_cmd, check=True)

print("Done.")
print("Output:", out_fp)

Minimum pixels for MMU: 56
0...10...20...30...40...50...60...70...80...90...100 - done in 00:07:03.
0...10...20...30...40...50...60...70...80...90...100 - done in 00:04:25.
Done.
Output: /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_8conn_5ha_1105.tif


In [10]:
import os
import math
import subprocess
import sys
import rasterio
import numpy as np

in_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist.tif"
tmp_binary_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/tmp_binary_gt08_TRUE_0_1.tif"
out_fp = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_gt08_4conn_5ha_TRUE_0_1.tif"

threshold = 0.8
pixel_size = 30
mmu_ha = 5
min_pixels = math.ceil((mmu_ha * 10000) / (pixel_size * pixel_size))

print("Minimum pixels:", min_pixels)

for fp in [tmp_binary_fp, out_fp]:
    if os.path.exists(fp):
        os.remove(fp)

# ------------------------------------------------------------
# 1) Create a real 0/1 raster
# ------------------------------------------------------------
with rasterio.open(in_fp) as src:
    profile = src.profile.copy()
    nodata_in = src.nodata

    profile.update(
        dtype="uint8",
        count=1,
        nodata=None,
        compress="lzw",
        tiled=True,
        BIGTIFF="YES"
    )

    with rasterio.open(tmp_binary_fp, "w", **profile) as dst:
        for _, window in src.block_windows(1):
            arr = src.read(1, window=window)

            # Everything > threshold becomes 1, everything else becomes 0
            out = np.where(arr > threshold, 1, 0).astype("uint8")

            # If the original input has NoData, force those pixels to 0 as well
            if nodata_in is not None:
                out[arr == nodata_in] = 0

            dst.write(out, 1, window=window)

print("Binary raster written:", tmp_binary_fp)

# ------------------------------------------------------------
# 2) Sieve small 1-patches smaller than 5 ha
# ------------------------------------------------------------
sieve_cmd = [
    sys.executable, "-m", "osgeo_utils.gdal_sieve",
    "-st", str(min_pixels),
    "-4",
    "-nomask",
    tmp_binary_fp,
    out_fp
]

print(" ".join(sieve_cmd))
subprocess.run(sieve_cmd, check=True)

# Remove NoData flag from output, just to be safe
subprocess.run(["gdal_edit.py", "-unsetnodata", out_fp], check=True)

print("Done.")
print("Output:", out_fp)

Minimum pixels: 56
Binary raster written: /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/tmp_binary_gt08_TRUE_0_1.tif
/mnt/dss_project/lmandl/anaconda3/envs/backcasting/bin/python -m osgeo_utils.gdal_sieve -st 56 -4 -nomask /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/tmp_binary_gt08_TRUE_0_1.tif /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_gt08_4conn_5ha_TRUE_0_1.tif
0...10...20...30...40...50...60...70...80...90...100 - done in 00:05:34.
Done.
Output: /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_gt08_4conn_5ha_TRUE_0_1.tif


In [11]:
subprocess.run(sieve_cmd, check=True)

subprocess.run(["gdal_edit.py", "-a_nodata", "0", out_fp], check=True)

print("Done.")
print("Output:", out_fp)

0...10...20...30...40...50...60...70...80...90...100 - done in 00:05:38.
Done.
Output: /mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_gt08_4conn_5ha_TRUE_0_1.tif


In [ ]:
## build slope vrt

from pathlib import Path


in_dir = Path("/mnt/eo/EO4Backcasting/_sen_slope_1985_1995/tile_slopes")
vrt_fp = in_dir.parent / "slope_NBR_1985_1995_mosaic.vrt"
out_fp = in_dir.parent / "slope_NBR_1985_1995_mosaic.tif"
filelist_fp = in_dir.parent / "slope_tile_list.txt"

tif_files = sorted(in_dir.glob("*.tif"))

print(f"Number of tif files: {len(tif_files)}")
print("First 5 files:")
for fp in tif_files[:5]:
    print(fp)

with open(filelist_fp, "w") as f:
    for fp in tif_files:
        f.write(str(fp) + "\n")

print("Wrote file list to:", filelist_fp)

In [ ]:
import subprocess

buildvrt_cmd = [
    "gdalbuildvrt",
    "-input_file_list", str(filelist_fp),
    str(vrt_fp)
]

subprocess.run(buildvrt_cmd, check=True)
print("VRT written to:", vrt_fp)

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# ----------------------------------
# FILE PATHS
# ----------------------------------
mask_fp  = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist_8conn_5ha_1105.tif"
prob_fp  = "/mnt/eo/EO4Backcasting/_predictions/predictions_global_model/global_model_no_dist.tif"
slope_fp = "/mnt/eo/EO4Backcasting/_sen_slope_1985_1995/slope_NBR_1985_1995_mosaic.vrt"

# ----------------------------------
# SETTINGS
# ----------------------------------
n_samples = 500000   # für ersten Test lieber kleiner starten
seed = 42

rng = np.random.default_rng(seed)

with rasterio.open(mask_fp) as mask_src, \
     rasterio.open(prob_fp) as prob_src, \
     rasterio.open(slope_fp) as slope_src:

    # nur Grid prüfen, CRS-Strings ignorieren
    if not (
        mask_src.width == prob_src.width == slope_src.width and
        mask_src.height == prob_src.height == slope_src.height and
        mask_src.transform == prob_src.transform == slope_src.transform
    ):
        raise ValueError("Die Raster liegen nicht exakt auf demselben Grid.")

    height = mask_src.height
    width = mask_src.width
    total_pixels = height * width

    sample_idx = rng.choice(total_pixels, size=n_samples, replace=False)
    rows, cols = np.unravel_index(sample_idx, (height, width))

    xs, ys = rasterio.transform.xy(mask_src.transform, rows, cols)
    pts = list(zip(xs, ys))

    # Werte mit Fortschrittsbalken lesen
    mask_vals = np.array(
        [v[0] for v in tqdm(mask_src.sample(pts), total=len(pts), desc="Sampling mask")]
    )
    prob_vals = np.array(
        [v[0] for v in tqdm(prob_src.sample(pts), total=len(pts), desc="Sampling probability")]
    )
    slope_vals = np.array(
        [v[0] for v in tqdm(slope_src.sample(pts), total=len(pts), desc="Sampling slope")]
    )

    valid = np.isfinite(mask_vals) & np.isfinite(prob_vals) & np.isfinite(slope_vals)

    if mask_src.nodata is not None:
        valid &= (mask_vals != mask_src.nodata)
    if prob_src.nodata is not None:
        valid &= (prob_vals != prob_src.nodata)
    if slope_src.nodata is not None:
        valid &= (slope_vals != slope_src.nodata)

df = pd.DataFrame({
    "mask": mask_vals[valid],
    "prob": prob_vals[valid],
    "slope": slope_vals[valid]
})

g1 = df[df["mask"] == 1].copy()
g2 = df[(df["mask"] != 1) & (df["prob"] >= 0) & (df["prob"] <= 0.3)].copy()

print("Sampled valid pixels:", len(df))
print("Group 1 (mask == 1):", len(g1))
print("Group 2 (mask != 1 & prob 0-0.3):", len(g2))

In [ ]:
# boxplots

plt.figure(figsize=(8, 6))
plt.boxplot(
    [g1["slope"].values, g2["slope"].values],
    labels=["mask == 1", "mask != 1\nprob 0-0.3"],
    showfliers=False
)

plt.ylabel("NBR slope")
plt.title("Distribution of NBR slope values")
plt.grid(axis="y", alpha=0.3)
plt.show()

